# US Wildfire Size Classification — Advanced Models

This notebook trains two higher-capacity models on top of the baselines from `baseline_models.ipynb`.

| # | Model | Purpose |
|---|---|---|
| 1 | **Random Forest** (class_weight='balanced') | Ensemble baseline, handles imbalance natively |
| 2 | **XGBoost** (SMOTE) | Gradient boosting with synthetic minority oversampling |
| 3 | **Comparison** | Both strategies evaluated side-by-side against baselines |

All models use the same `build_pipeline()` from `Preprocessing.py` and the same stratified 5-fold CV setup as the baseline notebook so results are directly comparable.

## 1  Imports

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV,
    cross_validate,
)
from sklearn.metrics import (
    make_scorer,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
import joblib

# XGBoost
from xgboost import XGBClassifier

# SMOTE — install with: pip install imbalanced-learn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Preprocessing module (one folder up)
sys.path.insert(0, "..")
from Preprocessing import build_pipeline, load_and_split, encode_labels

# ── Paths (pathlib — works on Windows/Lenovo without os.path) ──────────────
DATA_PATH   = Path("..") / "data" / "FW_Veg_Rem_Combined.csv"
OUTPUTS_DIR = Path("..") / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("All imports OK")

## 2  Load data

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}\n"
        "Download with:\n"
        "  pip install kagglehub\n"
        "  python -c \"import kagglehub, shutil; "
        "p = kagglehub.dataset_download('capcloudcoder/us-wildfire-data-plus-other-attributes'); "
        "shutil.copy(p + '/FW_Veg_Rem_Combined.csv', '../data/')\""
    )

X_train, X_val, X_test, y_train, y_val, y_test = load_and_split(DATA_PATH)

# Combine train + val for cross-validation (test set untouched)
X_cv = pd.concat([X_train, X_val], ignore_index=True)
y_cv = pd.concat([y_train, y_val], ignore_index=True)

# Encoded integer labels needed by XGBoost
y_train_enc, y_val_enc, y_test_enc, le = encode_labels(y_train, y_val, y_test)
y_cv_enc = np.concatenate([y_train_enc, y_val_enc])

print(f"CV pool : {len(X_cv):,} samples")
print(f"Test    : {len(X_test):,} samples (held out)")

## 3  Shared CV setup

Identical to the baseline notebook so all results are directly comparable.

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy":        "accuracy",
    "precision_macro": make_scorer(precision_score, average="macro", zero_division=0),
    "recall_macro":    make_scorer(recall_score,    average="macro", zero_division=0),
    "f1_macro":        make_scorer(f1_score,        average="macro", zero_division=0),
    "roc_auc_ovr":     make_scorer(
                           roc_auc_score,
                           response_method="predict_proba",
                           multi_class="ovr",
                           average="macro",
                       ),
}

METRIC_MAP = {
    "test_accuracy":        "Accuracy",
    "test_precision_macro": "Precision (macro)",
    "test_recall_macro":    "Recall (macro)",
    "test_f1_macro":        "F1 (macro)",
    "test_roc_auc_ovr":     "ROC-AUC (OVR macro)",
}

print(f"CV : StratifiedKFold(n_splits=5, shuffle=True, random_state=42)")
print(f"Metrics : {', '.join(METRIC_MAP.values())}")

## 4  Imbalance strategies

Class B (small fires) dominates at ~66% of records. Two strategies are compared:

| Strategy | How it works | Risk |
|---|---|---|
| `class_weight='balanced'` | Up-weights minority classes during loss computation | None — no data generated |
| **SMOTE** | Synthesises new minority-class samples from nearest neighbours | Must sit *inside* the CV pipeline to avoid leakage |

SMOTE is applied inside an `imblearn.Pipeline` so synthetic samples are only ever created from training folds — the validation fold is never seen during oversampling.

## 5  Random Forest — `class_weight='balanced'`

### 5.1  Hyperparameter search

`RandomizedSearchCV` with `n_iter=20` over the grid below.  
Budget justification: 20 × 3-fold inner CV = 60 fits — feasible in ~10 min on a laptop.  
Primary scoring metric: **F1 (macro)** — penalises ignoring minority classes.

In [ ]:
rf_param_grid = {
    # More trees reduce variance; 300 is a practical ceiling on a laptop
    "classifier__n_estimators":     [100, 200, 300],
    # None = fully grown; constraining depth regularises against overfitting
    "classifier__max_depth":        [None, 10, 20, 30],
    # min_samples_leaf smooths decision boundaries for minority classes
    "classifier__min_samples_leaf": [1, 2, 5],
    # max_features controls per-split randomness — sqrt is the RF default
    "classifier__max_features":     ["sqrt", "log2"],
}

rf_pipeline = build_pipeline(
    RandomForestClassifier(
        class_weight="balanced",  # handles imbalance — no data leakage
        random_state=42,
        n_jobs=-1,
    )
)

rf_search = RandomizedSearchCV(
    rf_pipeline,
    rf_param_grid,
    n_iter=20,
    cv=3,                   # inner CV for tuning
    scoring="f1_macro",
    n_jobs=-1,
    random_state=42,
    verbose=1,
    refit=True,             # refit best params on full X_cv
)

print("Searching Random Forest hyperparameters ...")
rf_search.fit(X_cv, y_cv)

print(f"\nBest params : {rf_search.best_params_}")
print(f"Best CV F1  : {rf_search.best_score_:.4f}")

### 5.2  Outer 5-fold CV evaluation

In [ ]:
print("Running outer 5-fold CV for Random Forest ...")
rf_cv_raw = cross_validate(
    rf_search.best_estimator_,
    X_cv, y_cv,
    cv=CV,
    scoring=scoring,
    n_jobs=-1,
)
print("Done.")

for key, label in METRIC_MAP.items():
    scores = rf_cv_raw[key]
    print(f"  {label:<22s}: {scores.mean():.4f} ± {scores.std():.4f}")

## 6  XGBoost — SMOTE oversampling

### 6.1  Hyperparameter search

SMOTE is placed *before* XGBoost inside an `imblearn.Pipeline`.  
This guarantees synthetic samples are generated only from each training fold — the validation fold is never used in oversampling, preventing leakage.

XGBoost requires integer labels, so we use `y_cv_enc` (0–5) from `encode_labels()`.

In [ ]:
xgb_param_grid = {
    # More rounds; early stopping not used here to keep CV simple
    "classifier__n_estimators":     [100, 200, 300],
    # Shallow trees reduce overfitting on tabular data
    "classifier__max_depth":        [3, 5, 7],
    # Lower LR = more conservative updates
    "classifier__learning_rate":    [0.05, 0.1, 0.2],
    # min_child_weight ~ min_samples_leaf; regularises minority-class splits
    "classifier__min_child_weight": [1, 3, 5],
    # Subsample adds stochasticity, reduces variance
    "classifier__subsample":        [0.8, 1.0],
}

# imblearn Pipeline: SMOTE runs inside each CV fold (after preprocessor)
xgb_pipeline = ImbPipeline([
    ("preprocessor", build_pipeline().named_steps["preprocessor"]),
    ("smote",        SMOTE(random_state=42)),
    ("classifier",   XGBClassifier(
                         eval_metric="mlogloss",
                         random_state=42,
                         n_jobs=-1,
                         verbosity=0,
                     )),
])

xgb_search = RandomizedSearchCV(
    xgb_pipeline,
    xgb_param_grid,
    n_iter=20,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    random_state=42,
    verbose=1,
    refit=True,
)

print("Searching XGBoost hyperparameters (with SMOTE) ...")
xgb_search.fit(X_cv, y_cv_enc)

print(f"\nBest params : {xgb_search.best_params_}")
print(f"Best CV F1  : {xgb_search.best_score_:.4f}")

### 6.2  Outer 5-fold CV evaluation

In [ ]:
print("Running outer 5-fold CV for XGBoost + SMOTE ...")
xgb_cv_raw = cross_validate(
    xgb_search.best_estimator_,
    X_cv, y_cv_enc,
    cv=CV,
    scoring=scoring,
    n_jobs=-1,
)
print("Done.")

for key, label in METRIC_MAP.items():
    scores = xgb_cv_raw[key]
    print(f"  {label:<22s}: {scores.mean():.4f} ± {scores.std():.4f}")

## 7  Model comparison

Summary table combining both advanced models.  
Paste the baseline results from `baseline_models.ipynb` into `baseline_results` below once you have them, so the full comparison is in one place.

In [ ]:
# ── Collect results ─────────────────────────────────────────────────────────
adv_cv_results = {
    "Random Forest (balanced)": rf_cv_raw,
    "XGBoost (SMOTE)":          xgb_cv_raw,
}

summary_rows = []
for model_name, raw in adv_cv_results.items():
    row = {"Model": model_name}
    for key, label in METRIC_MAP.items():
        scores = raw[key]
        row[label] = f"{scores.mean():.4f} ± {scores.std():.4f}"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("Model")

divider = "=" * 95
print(divider)
print("  Advanced Models — Stratified 5-Fold CV Results   (mean ± std)")
print(divider)
print(summary_df.to_string())
print(divider)

summary_df

### 7.1  Visualisation — advanced vs baseline comparison

In [ ]:
# ── Mean scores for plotting ─────────────────────────────────────────────────
mean_df = pd.DataFrame(
    {
        label: [adv_cv_results[m][key].mean() for m in adv_cv_results]
        for key, label in METRIC_MAP.items()
    },
    index=list(adv_cv_results.keys()),
)

std_df = pd.DataFrame(
    {
        label: [adv_cv_results[m][key].std() for m in adv_cv_results]
        for key, label in METRIC_MAP.items()
    },
    index=list(adv_cv_results.keys()),
)

# ── Bar chart ────────────────────────────────────────────────────────────────
metrics_to_plot = ["Accuracy", "F1 (macro)", "ROC-AUC (OVR macro)"]
x     = np.arange(len(metrics_to_plot))
width = 0.3
colors = ["#2196F3", "#FF5722"]

fig, ax = plt.subplots(figsize=(9, 5))
for i, (model_name, color) in enumerate(zip(adv_cv_results.keys(), colors)):
    means = [mean_df.loc[model_name, m] for m in metrics_to_plot]
    stds  = [std_df.loc[model_name, m]  for m in metrics_to_plot]
    ax.bar(
        x + i * width, means, width,
        label=model_name, color=color, alpha=0.85,
        edgecolor="white",
        yerr=stds, capsize=4, error_kw={"elinewidth": 1.5},
    )

ax.set_xticks(x + width / 2)
ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.set_ylabel("Score", fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_title("Advanced Models Comparison  (mean ± std, 5-fold CV)", fontsize=13)
ax.legend(fontsize=10)
ax.axhline(1 / 6, color="gray", linestyle=":", linewidth=1)
ax.text(x[-1] + width + 0.15, 1/6 + 0.01, "random (1/6)", color="gray", fontsize=8)
sns.despine()
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "advanced_model_comparison.png", dpi=150)
plt.show()
print("Saved → outputs/advanced_model_comparison.png")

### 7.2  Heatmap of all metrics

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.5))
sns.heatmap(
    mean_df,
    annot=True, fmt=".4f",
    cmap="RdYlGn", vmin=0, vmax=1,
    linewidths=0.5, linecolor="white",
    ax=ax,
)
ax.set_title("Advanced Models — Mean CV Scores (5-fold stratified)", fontsize=13, pad=12)
ax.set_ylabel("")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "advanced_cv_heatmap.png", dpi=150)
plt.show()
print("Saved → outputs/advanced_cv_heatmap.png")

## 8  Save best models

Both pipelines are serialised so Neera can load the best one directly for SHAP analysis without retraining.

In [ ]:
joblib.dump(rf_search.best_estimator_,  OUTPUTS_DIR / "best_model_rf.pkl")
joblib.dump(xgb_search.best_estimator_, OUTPUTS_DIR / "best_model_xgb.pkl")

# Also save the label encoder so Neera can decode XGBoost predictions
joblib.dump(le, OUTPUTS_DIR / "label_encoder.pkl")

print("Saved:")
print("  outputs/best_model_rf.pkl")
print("  outputs/best_model_xgb.pkl")
print("  outputs/label_encoder.pkl")

## 9  Key takeaways

**Random Forest (`class_weight='balanced'`)**  
Reweighting minority classes during training is the simplest and safest imbalance correction — no synthetic data, zero leakage risk. The ensemble averaging also reduces the high variance seen in the single Decision Tree baseline.

**XGBoost + SMOTE**  
SMOTE generates synthetic minority-class samples in feature space before each training fold, giving XGBoost a more balanced training distribution. Placing SMOTE inside the `imblearn.Pipeline` is critical — oversampling outside the CV loop would allow validation-fold information to influence the synthetic samples, introducing leakage.

**Imbalance strategy comparison**  
Compare the F1 (macro) and ROC-AUC of both models against the baseline Decision Tree: any improvement on minority classes (C–G) at the cost of majority-class (B) accuracy is an acceptable trade-off given the real-world cost of missing large fires.

**Next steps (Neera — Interpretation)**
- Load `best_model_rf.pkl` or `best_model_xgb.pkl` from `outputs/`
- Run SHAP on the best performer
- Analyse the confusion matrix for false negatives on large-fire classes (E, F, G)